# 01 — Data Collection

Collects daily **PETR4.SA** market data for the last five years from Yahoo Finance and saves the untransformed data to `data/raw`. Requires `pandas` and `yfinance`.

In [ ]:
from pathlib import Path

import pandas as pd
import yfinance as yf

TICKER = "PETR4.SA"
PERIOD = "5y"
INTERVAL = "1d"

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "petr4_daily.csv"

In [ ]:
df_raw = yf.Ticker(TICKER).history(
    period=PERIOD,
    interval=INTERVAL,
    auto_adjust=False,
    actions=True,
)

if df_raw.empty:
    raise RuntimeError(f"No data returned for {TICKER}.")

df_raw.index = pd.to_datetime(df_raw.index).tz_localize(None)
df_raw.index.name = "Date"
df_raw = df_raw.sort_index()
df_raw.head()

In [ ]:
required_columns = {"Open", "High", "Low", "Close", "Volume"}
missing_columns = required_columns.difference(df_raw.columns)

if missing_columns:
    raise ValueError(f"Missing columns: {sorted(missing_columns)}")
if not df_raw.index.is_unique:
    raise ValueError("Duplicate dates were found.")

pd.Series({
    "records": len(df_raw),
    "start_date": df_raw.index.min().date(),
    "end_date": df_raw.index.max().date(),
    "missing_values": int(df_raw.isna().sum().sum()),
})

In [ ]:
RAW_PATH.parent.mkdir(parents=True, exist_ok=True)
df_raw.to_csv(RAW_PATH)

print(f"Data saved to: {RAW_PATH}")